### [Predictive Power of Candlestick Patterns in Stock Trading - Part 1](https://medium.com/@Tobi_Lux/d71dd92b4b27)

```shell
# importing TA-Lib
url = 'https://anaconda.org/conda-forge/libta-lib/0.4.0/download/linux-64/libta-lib-0.4.0-h166bdaf_1.tar.bz2'
!curl -L $url | tar xj -C /usr/lib/x86_64-linux-gnu/ lib --strip-components=1
url = 'https://anaconda.org/conda-forge/ta-lib/0.4.19/download/linux-64/ta-lib-0.4.19-py310hde88566_4.tar.bz2'
!curl -L $url | tar xj -C /usr/local/lib/python3.10/dist-packages/ lib/python3.10/site-packages/talib --strip-components=3
```

In [1]:
print("⏳ Downloading TA-Lib 0.6.4 from GitHub...")
!wget -q https://github.com/TA-Lib/ta-lib/releases/download/v0.6.4/ta-lib-0.6.4-src.tar.gz

print("📦 Extracting...")
!tar -xzf ta-lib-0.6.4-src.tar.gz > /dev/null 2>&1 && rm -rf ta-lib-0.6.4-src.tar.gz

print("🔧 Building TA-Lib C library...")
!cd ta-lib-0.6.4 && ./configure --prefix=/usr > /dev/null 2>&1 && make > /dev/null 2>&1 && make install > /dev/null 2>&1

print("🐍 Installing Python wrapper...")
!pip install -q TA-Lib

print("✅ Done! TA-Lib 0.6.4 is ready to use!")

⏳ Downloading TA-Lib 0.6.4 from GitHub...
📦 Extracting...
🔧 Building TA-Lib C library...
🐍 Installing Python wrapper...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 23.1 MB/s eta 0:00:00
✅ Done! TA-Lib 0.6.4 is ready to use!


In TA-Lib, each candlestick pattern detection function provides daily outputs as follows:
```
output = (200) or 100 → (strong) bullish signal

output = 0 → no signal

output = (-200) or -100 → (strong) bearish signal
```

In [2]:
import sys
import platform
IN_COLAB = 'google.colab' in sys.modules

from IPython.display import display

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from datetime import date
import yfinance as yf
from tqdm import tqdm

import scipy
from scipy import stats as st
import statsmodels.api as sm

import talib

%autosave 10

print(f"Python: {sys.version} [{platform.architecture()}]")
print(f"Numpy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"YFinance: {yf.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Statsmodels: {sm.__version__}")
print(f"TA-Lib: {talib.__version__}")

Autosaving every 10 seconds
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0] [('64bit', 'ELF')]
Numpy: 2.0.2
Pandas: 2.2.2
YFinance: 0.2.66
SciPy: 1.16.3
Statsmodels: 0.14.6
TA-Lib: 0.6.8


In [3]:
start, end = '2000-01-01', '2024-07-05'
prices = yf.download('^GDAXI', start='2000-01-01', end='2024-07-05', progress=False)

if isinstance(prices.columns, pd.MultiIndex):
    if prices.columns.nlevels > 1:
        prices.columns = prices.columns.get_level_values(0)
display(prices.sample(15))

Price,Close,High,Low,Open,Volume
Date,,,,,
2023-01-24,15093.110352,15147.450195,15022.549805,15140.500000,48335400
2018-06-07,12811.049805,12914.849609,12760.540039,12877.849609,94068000
2024-01-17,16431.689453,16435.679688,16345.019531,16400.419922,71437800
2004-09-07,3889.040039,3905.110107,3873.530029,3888.540039,71858600
2009-05-18,4851.959961,4858.649902,4655.529785,4702.810059,119160500
2024-02-28,17601.220703,17605.609375,17552.980469,17569.609375,56644500
2001-06-18,5869.040039,5948.839844,5853.069824,5921.609863,49771700
2001-09-07,4730.669922,4910.930176,4713.120117,4871.410156,86180900
2003-05-20,2838.929932,2875.399902,2820.409912,2848.189941,105582800


In [4]:
# identification of candle patterns
candle_names = talib.get_function_groups()['Pattern Recognition']
print(candle_names)

# defining the look_ahead-intervall and forming reduced price dataframe without last day_look_ahead-prices
days_look_ahead = 20
days = list(range(days_look_ahead + 1))
idx = prices.index[0:-(days_look_ahead + 1)]
prices_red = prices[prices.index.isin(idx)]

display(prices_red.sample(10))

# counting of occurence of signals per candle pattern
cp = pd.DataFrame(index=prices_red.index)
signal_ = [-200, -100, 100, 200]
cp_stat = pd.DataFrame(columns=candle_names, index=signal_)

# nb of occurance of (candles, type)-combinations
for candle in candle_names:
  cp[candle] = getattr(talib, candle)(prices_red.Open, prices_red.High, prices_red.Low, prices_red.Close)
  for sg in signal_:
    cp_stat.loc[sg, candle] = cp[cp[candle] == sg][candle].count()

# occurence per pattern (bullish and bearish)
cp_stat_tr = cp_stat.T
cp_stat_tr['sum_bull'] = cp_stat_tr.iloc[:,2] + cp_stat_tr.iloc[:,3]
cp_stat_tr['sum_bear'] = cp_stat_tr.iloc[:,0] + cp_stat_tr.iloc[:,1]
cp_stat_tr['sum_all'] = cp_stat_tr['sum_bull'] + cp_stat_tr['sum_bear']
cp_stat_tr.sort_values('sum_all', inplace=True, ascending=False)

display(cp_stat_tr)

['CDL2CROWS', 'CDL3BLACKCROWS', 'CDL3INSIDE', 'CDL3LINESTRIKE', 'CDL3OUTSIDE', 'CDL3STARSINSOUTH', 'CDL3WHITESOLDIERS', 'CDLABANDONEDBABY', 'CDLADVANCEBLOCK', 'CDLBELTHOLD', 'CDLBREAKAWAY', 'CDLCLOSINGMARUBOZU', 'CDLCONCEALBABYSWALL', 'CDLCOUNTERATTACK', 'CDLDARKCLOUDCOVER', 'CDLDOJI', 'CDLDOJISTAR', 'CDLDRAGONFLYDOJI', 'CDLENGULFING', 'CDLEVENINGDOJISTAR', 'CDLEVENINGSTAR', 'CDLGAPSIDESIDEWHITE', 'CDLGRAVESTONEDOJI', 'CDLHAMMER', 'CDLHANGINGMAN', 'CDLHARAMI', 'CDLHARAMICROSS', 'CDLHIGHWAVE', 'CDLHIKKAKE', 'CDLHIKKAKEMOD', 'CDLHOMINGPIGEON', 'CDLIDENTICAL3CROWS', 'CDLINNECK', 'CDLINVERTEDHAMMER', 'CDLKICKING', 'CDLKICKINGBYLENGTH', 'CDLLADDERBOTTOM', 'CDLLONGLEGGEDDOJI', 'CDLLONGLINE', 'CDLMARUBOZU', 'CDLMATCHINGLOW', 'CDLMATHOLD', 'CDLMORNINGDOJISTAR', 'CDLMORNINGSTAR', 'CDLONNECK', 'CDLPIERCING', 'CDLRICKSHAWMAN', 'CDLRISEFALL3METHODS', 'CDLSEPARATINGLINES', 'CDLSHOOTINGSTAR', 'CDLSHORTLINE', 'CDLSPINNINGTOP', 'CDLSTALLEDPATTERN', 'CDLSTICKSANDWICH', 'CDLTAKURI', 'CDLTASUKIGAP', 'CDL

Price,Close,High,Low,Open,Volume
Date,,,,,
2017-03-20,12052.900391,12082.309570,12033.240234,12050.809570,107350200
2011-01-31,7077.479980,7107.370117,7033.089844,7094.560059,93018500
2016-05-20,9916.019531,9921.589844,9852.709961,9878.490234,78255500
2018-05-03,12690.150391,12798.160156,12665.330078,12775.660156,98817100
2020-01-16,13429.429688,13492.740234,13382.980469,13463.459961,39611800
2005-07-15,4712.899902,4723.649902,4691.200195,4702.000000,102995000
2005-11-22,5174.720215,5182.729980,5161.350098,5178.959961,97497500
2015-07-15,11539.660156,11566.019531,11475.959961,11508.339844,66692300
2021-02-09,14011.799805,14061.610352,13962.139648,14055.599609,58414500


,-200,-100,100,200,sum_bull,sum_bear,sum_all
CDLSPINNINGTOP,0,672,685,0,685,672,1357
CDLLONGLINE,0,553,759,0,759,553,1312
CDLBELTHOLD,0,563,616,0,616,563,1179
CDLCLOSINGMARUBOZU,0,354,571,0,571,354,925
CDLLONGLEGGEDDOJI,0,0,852,0,852,0,852
...,...,...,...,...,...,...,...
CDLKICKING,0,0,0,0,0,0,0
CDLKICKINGBYLENGTH,0,0,0,0,0,0,0
CDLMATHOLD,0,0,0,0,0,0,0
CDLCONCEALBABYSWALL,0,0,0,0,0,0,0


In [5]:
# pattern detection (example harami-pattern)
harami = talib.CDLHARAMI(prices.Open, prices.High, prices.Low, prices.Close)

# identifying bullish/bearis signals and preparation for
bull_harami_plot = np.where(harami>0, prices.Close, np.nan)
bear_harami_plot = np.where(harami<0, prices.Close, np.nan)

In [6]:
# PLotting of Data
fig = go.Figure()
fig.add_trace(
    go.Candlestick(
        x=prices.index, open=prices['Open'],
        high=prices['High'], low=prices['Low'], close=prices['Close'],
        )
    )
fig.add_trace(
    go.Scatter(
        x=prices.index, y=bull_harami_plot,
        mode='markers', marker_symbol='triangle-up', marker_color='yellow', marker_size=15,
        name='bull_harami'
        )
    )
fig.add_trace(
    go.Scatter(
        x=prices.index, y=bear_harami_plot,
        mode='markers', marker_symbol='triangle-down', marker_color='lightskyblue', marker_size=15,
        name='bear_harami',
        )
    )

fig.update_layout(
    template='plotly_dark',
    autosize=False,
    width=1200, height=600,
    title = f'DAX between {start} to {end}',
    xaxis_title='date', yaxis_title='prices',
    xaxis_rangeslider_visible=False
    )

fig.show()

In [7]:
# Bar-Chart showing the occurence of all patterns
main_title = 'number of occurences of all candlestick-pattterns'
sub_title = f'instrument: DAX,  duration: {start} - {end}'
title = main_title + '<br><br><sup>' + sub_title + '</sup>'

# plotting
fig = go.Figure()
for i in range(4):
  y_x = (cp_stat_tr.iloc[:, i].tolist())
  fig.add_trace(go.Bar(x=candle_names, y=y_x, name = cp_stat_tr.columns[i]))
fig.update_layout(template='plotly_dark', autosize=False, width=1200, height=600)
fig.update_layout(title=title, xaxis_title='candle-pattern', yaxis_title='# of occurences', legend_title="Signals from TA-LIB")
fig.update_layout(barmode='stack')
fig.update_xaxes(tickangle= -90)
fig.show()

In [8]:
# most active candle pattern
nb_occ = 100
most_active_bull = cp_stat_tr[cp_stat_tr['sum_bull'] > nb_occ].sort_values(by=['sum_bull'], ascending=False)
most_active_bear = cp_stat_tr[cp_stat_tr['sum_bear'] > nb_occ].sort_values(by=['sum_bear'], ascending=False)

In [9]:
display(most_active_bull)

,-200,-100,100,200,sum_bull,sum_bear,sum_all
CDLLONGLEGGEDDOJI,0,0,852,0,852,0,852
CDLDOJI,0,0,852,0,852,0,852
CDLLONGLINE,0,553,759,0,759,553,1312
CDLSPINNINGTOP,0,672,685,0,685,672,1357
CDLRICKSHAWMAN,0,0,670,0,670,0,670
CDLBELTHOLD,0,563,616,0,616,563,1179
CDLCLOSINGMARUBOZU,0,354,571,0,571,354,925
CDLSHORTLINE,0,295,423,0,423,295,718
CDLHIGHWAVE,0,390,396,0,396,390,786
CDLHARAMI,0,250,295,0,295,250,545


In [10]:
display(most_active_bear)

,-200,-100,100,200,sum_bull,sum_bear,sum_all
CDLSPINNINGTOP,0,672,685,0,685,672,1357
CDLBELTHOLD,0,563,616,0,616,563,1179
CDLLONGLINE,0,553,759,0,759,553,1312
CDLHIKKAKE,82,314,219,69,288,396,684
CDLHIGHWAVE,0,390,396,0,396,390,786
CDLCLOSINGMARUBOZU,0,354,571,0,571,354,925
CDLSHORTLINE,0,295,423,0,423,295,718
CDLHARAMI,0,250,295,0,295,250,545
CDLENGULFING,0,241,200,0,200,241,441
CDLMARUBOZU,0,146,253,0,253,146,399


In [11]:
# Bar-Chart showing the most active patterns
modes = ['bullish','bearish']
main_title1 = f'number of occurences of most active candle patterns bullish mode'
main_title2 = f'number of occurences of most active candle patterns bearish mode'
sub_title = f'instrument: DAX,  duration: {start} - {end}'
title1 = main_title1 + '<br><br><sup>' + sub_title + '</sup>'
title2 = main_title2 + '<br><br><sup>' + sub_title + '</sup>'
text_subtitles = [main_title1, main_title2]

# plotting
fig = make_subplots(rows=2, cols=1, subplot_titles = (text_subtitles))
for i, mode in enumerate(modes):
  if mode =='bullish':
    y_x = most_active_bull['sum_bull']
    x_x = most_active_bull.index
  elif mode =='bearish':
    y_x = most_active_bear['sum_bear']
    x_x = most_active_bear.index
  fig.add_trace(go.Bar(x=x_x, y=y_x, name=f'{modes[i]} candlestick patterns'), row=i+1, col=1)
  fig.update_xaxes(title_text='candlestick patterns', row=i+1, col=1)
  fig.update_yaxes(title_text='nb of occurence', row=i+1, col=1)

fig.update_layout(template='plotly_dark', autosize=False, width=1200, height=1200)
fig.update_layout(legend_title="Signals from TA-LIB")
fig.update_xaxes(tickangle= -90)
fig.show()

In [12]:
# calculations of development of positions (=start at day of detection of a pattern (=occurence) until + days_look_ahead)
# going LONG/SHORT depending on the signal
# recording for each position from opening to close: price, returns and cumulated returns
# results collected in sortable dataframe

# preparation
traject_bull = pd.DataFrame()
traject_bear = pd.DataFrame()
trade_id_bull = 0
trade_id_bear = 0
indxs = prices.index.tolist()

# looking at bullish and bearish pattern separately
for mode in modes:
  # separate analyses for bullish and bearish candlestick  pattern
  if mode =='bullish':
    top_candle_names = most_active_bull.index.tolist()
  elif mode =='bearish':
    top_candle_names = most_active_bear.index.tolist()

  # looping through all bullish and bearish candlestick patterns separately
  for candle in tqdm(top_candle_names):
    # identification of dates and indexes of occurence of a certain "candlestick pattern"
    if mode == 'bullish':
      x_p = cp[cp[candle] > 0][candle]
    elif mode == 'bearish':
      x_p = cp[cp[candle] < 0][candle]
    idx = x_p.index.tolist()

    # looping through the identfied occurences
    for id  in idx:
      # for each occurence defining the ovservation intervall and collecting the corresponding prices
      id_start = indxs.index(id)
      id_x = indxs[id_start: id_start + days_look_ahead + 1]
      pr_x = prices[prices.index.isin(id_x)]['Close']
      if mode == 'bullish':
        trade_id_bull += 1
        rets_bull_x = pr_x.pct_change().fillna(0)              # positive return, since we are in a LONG positions
        cumrets_bull_x = (1 + rets_bull_x).cumprod()
        # transfering into a data-frame and connecting to the previous data
        dict_df =  {'pattern': [candle] * (days_look_ahead + 1),
                    'mode': ['bullish'] * (days_look_ahead + 1),
                    'pattern_id':[trade_id_bull] * (days_look_ahead + 1),
                    'date_id': id_x,
                    'day': days,
                    'close': pr_x.to_list(),
                    'rets':rets_bull_x.to_list(),
                    'cumrets':cumrets_bull_x.to_list()}
        tr_bull_x = pd.DataFrame(dict_df)
        traject_bull = pd.concat([traject_bull, tr_bull_x], axis=0)
      elif mode == 'bearish':
        trade_id_bear += 1
        rets_bear_x = - pr_x.pct_change().fillna(0)              # negative retun, since we are in a SHORT position
        cumrets_bear_x = (1 + rets_bear_x).cumprod()
        # transfering into a data-frame and connecting to the previous data
        dict_df =  {'pattern': [candle] * (days_look_ahead + 1),
                    'mode': ['bearish'] * (days_look_ahead + 1),
                    'pattern_id':[trade_id_bear] * (days_look_ahead + 1),
                    'date_id': id_x,
                    'day': days,
                    'close': pr_x.to_list(),
                    'rets':rets_bear_x.to_list(),
                    'cumrets':cumrets_bear_x.to_list()}
        tr_bear_x = pd.DataFrame(dict_df)
        traject_bear = pd.concat([traject_bear, tr_bear_x], axis=0)

100%|██████████| 13/13 [00:17<00:00,  1.38s/it]


In [13]:
# tabulation of averaged cumulated returns for each bullish/bearish candlestick pattern and each day

# preparation
days_str = ['day+' + str(i).zfill(2) for i in range(len(days))]
results_bull_avg_day = pd.DataFrame()
results_bear_avg_day = pd.DataFrame()

# separate analyses per bullish or bearish candlestick pattern per day
for day in days:
  tr_bull = traject_bull[(traject_bull['day'] == day)].groupby(['pattern'])['cumrets'].mean()
  results_bull_avg_day = pd.concat([results_bull_avg_day, tr_bull], axis=1)
  tr_bear = traject_bear[(traject_bear['day'] == day)].groupby(['pattern'])['cumrets'].mean()
  results_bear_avg_day = pd.concat([results_bear_avg_day, tr_bear], axis=1)

# renaming columns
results_bull_avg_day.columns = days_str
results_bear_avg_day.columns = days_str

In [14]:
# displaying the average position development for the top active bullish and bearish candlestick patterns

main_title1 = f'development of PnL (= cumulative returns) of most active bullish candlestick patterns'
main_title2 = f'development of PnL (= cumulative returns) of most active bearish candlestick patterns'
sub_title = 'instrument: DAX,  duration: 2000-01-01 - 2024-06-09'
title1 = main_title1 + '<br><br><sup>' + sub_title + '</sup>'
title2 = main_title2 + '<br><br><sup>' + sub_title + '</sup>'
text_subtitles = [title1, title2]

# plotting of cumulated returns per pattern and day as store in results_bull_avg_day and results_bear_avg_day
fig = make_subplots(rows=2, cols=1, subplot_titles = (text_subtitles))
for i in range(len(results_bull_avg_day)):
  y_x = results_bull_avg_day.iloc[i].tolist()
  fig.add_trace(go.Scatter(x=days_str, y=y_x, name=f'{results_bull_avg_day.index[i]} bullish'), row=1, col=1)
for i in range(len(results_bear_avg_day)):
  y_x = results_bear_avg_day.iloc[i].tolist()
  fig.add_trace(go.Scatter(x=days_str, y=y_x, name=f'{results_bear_avg_day.index[i]} bearish'), row=2, col=1)
fig.update_layout(template='plotly_dark', autosize=False, width=1200, height=900)
fig.update_xaxes(tickangle= -90)
fig.show()

In [15]:
# days for further analyses
days_analyzed = [1, 5, 10, 15, 20]

# extraction of the top performers by analysing the cumrets at selected days

# preparation
top_bull_px = []
top_bear_px = []
selected_cols = []

# extration of data for the days to be analyzed
for day in days_analyzed:
  selected_cols.append('day+' + str(day).zfill(2))
results_bull_selected = results_bull_avg_day[selected_cols]
results_bear_selected = results_bear_avg_day[selected_cols]

# selection ot the top 3 pattern for each analyzed day and collecting the data in a list
for col in selected_cols:
  x_bull = results_bull_selected[col].sort_values(ascending=False).iloc[0:3]
  y_bull = x_bull.index.tolist()
  top_bull_px = top_bull_px + y_bull
  x_bear = results_bear_selected[col].sort_values(ascending=False).iloc[0:3]
  y_bear = x_bear.index.tolist()
  top_bear_px = top_bear_px + y_bear

# removal of duplication from the lists
top_bull_patterns = list(set(top_bull_px))
top_bear_patterns = list(set(top_bear_px))

# printing results
print(f'top profitable bullish candlestick patterns: {top_bull_patterns}')
print(f'top profitable bearish candlestick patterns: {top_bear_patterns}')

top profitable bullish candlestick patterns: ['CDLCLOSINGMARUBOZU', 'CDLHIKKAKE', 'CDLHARAMI', 'CDLSHORTLINE', 'CDLHAMMER', 'CDLENGULFING']
top profitable bearish candlestick patterns: ['CDLDOJISTAR', 'CDLHIKKAKE', 'CDLMARUBOZU', 'CDLSHORTLINE', 'CDLLONGLINE', 'CDLHANGINGMAN']


In [16]:
# selection of pattern and mode for detailed analyses
candle = top_bull_patterns[2]
mode = modes[0]

# development of distribution of PnL (= cumulated returns) for selected patterns over time - BOX-plot
main_title = f'development of PnL distribution of positions after occurence of {candle} in {mode} mode'
sub_title = f'instrument: DAX,  duration: 2000-01-01 - 2024-06-09 '
title = main_title + '<br><br><sup>' + sub_title + '</sup>'

# preparation
if mode == 'bullish':
  p_x = traject_bull[(traject_bull['pattern'] == candle)]
elif mode == 'bearish':
  p_x = traject_bear[(traject_bear['pattern'] == candle)]

# plotting
fig = px.box(p_x, x='day', y ='cumrets', points='all')
fig.update_layout(template='plotly_dark', autosize=False, width=1200, height=600)
fig.update_layout(title=title, xaxis_title='days after occurence of pattern', yaxis_title='# PnL of position', legend_title='candle pattern')
fig.update_xaxes(dtick=1, tickangle= -90)
fig.show()

In [17]:
# distribution of PnL (=cumulated returns) over time

#preparations
days_str_analyzed = [f'day+' + str(i).zfill(2) for i in days_analyzed]

# plotting
fig = make_subplots(rows=1, cols=5, shared_yaxes=True, subplot_titles = (selected_cols))

for i, day in enumerate(days_analyzed):
  if mode == 'bullish':
    tx = traject_bull[(traject_bull['pattern'] == candle) & (traject_bull['day'] == day)]['cumrets']
  elif mode =='bearish':
    tx = traject_bear[(traject_bear['pattern'] == candle) & (traject_bear['day'] == day)]['cumrets']

  fig.add_trace(go.Histogram(y=tx.to_list(), name=days_str_analyzed[i], ybins=dict(start=0.75, end=1.25, size=0.01)), row=1, col=i+1)
  fig.add_hline(y=tx.mean(), line_dash="dot", row=1, col=i+1, annotation_text=f'mean: {tx.mean():.4f}', annotation_position="bottom right")

main_title = f'PnL-distribution after x-days of occurences of {candle} in {mode} mode, number of occurences: {len(tx)}'
title = main_title

fig.update_layout(template='plotly_dark', autosize=False, width=1600, height=600)
fig.update_layout(title=title, xaxis_title='# of occurences', yaxis_title='PnL of the day', legend_title='days after occurence')
fig.show()

In [18]:
def norm_test(data, method, alpha):
  ''' test of null-hypothesis: data is normally distributed
  Args:
    data: to be anlyzed
    method: test method to be applied
    alpha: confidence level
  Returns:
    the test result: 1 in case the null-hypothesis is confirmed and 0 if null-hyothesis is rejected
  Raises:
    nothing
  '''
  res, statistic, pvalue = 0, 0, 0
  # perform DAgoiostino & Pearson Test for Normality
  if method == 'dAgostinoPearson':
    statistic, pvalue = st.normaltest(data)
    res = 1 if pvalue > alpha else 0
  # perform Jarque-Brera-Test for Normality
  elif method == 'JarqueBera':
    statistic, pvalue = st.jarque_bera(data)
    res = 1 if pvalue > alpha else 0
  # Perform Lilliefors-Test for Normality
  elif method == 'Lilliefors':
    statistic, pvalue = sm.stats.diagnostic.lilliefors(data)
    res = 1 if pvalue > alpha else 0
  # perform Shapiro-Wilk test for Normality
  elif method == 'ShapiroWilk':
    statistic, pvalue = st.shapiro(data)
    res = 1 if pvalue > alpha else 0
  # Perform Anderson-Darling test for Normality
  elif method == 'AndersonDarling':
    rx = st.anderson(data)
    idx = np.where(rx.significance_level == alpha * 100)
    res = 1 if rx.statistic < rx.critical_values[idx] else 0
  elif method =='KolmogorovSmirnov':
    statistic, pvalue = sm.stats.diagnostic.kstest_normal(data)
    res = 1 if pvalue > alpha else 0
  res =1 if res>0 else 0
  return int(res)

In [19]:
# test for a normal-distribution of cummulated returns for different combination of (candle-patterns, mode) and a certain day
# using different statistical tests
test_names = ['Lilliefors', 'ShapiroWilk', 'JarqueBera']
alpha = 0.05

# preparations
if mode == 'bullish':
  top_patterns = top_bull_patterns
  traject_1 = traject_bull
elif mode == 'bearish':
  top_patterns = top_bear_patterns
  traject_1 = traject_bear
test_results = []

# execution of different tests for top-candle patternd for different days
for test in test_names:
  result_x = np.zeros((len(top_patterns), len(days_analyzed)))
  for i, candle in enumerate(top_patterns):
    tr1 = traject_1[traject_1['pattern'] == candle]
    for j, day in enumerate(days_analyzed):
      data = tr1[tr1['day'] == day]['cumrets']
      result_x[i,j] = norm_test(data, test, alpha)
  test_results =  test_results + [result_x]

In [20]:
# plotting of an overview about the test results according to different methods

# preparations
test_str = ['Test: ' + str(test_names[i]) for i in range(len(test_names))]
fig = make_subplots(rows=1, cols=len(test_names), shared_yaxes=True, shared_xaxes=True, subplot_titles = (test_str))

#sub-plotting
for k, test in enumerate(test_names):
  row = 1 # int(k/3) + 1
  col = k%3 + 1
  fig.add_trace(go.Heatmap(z=test_results[k], zmin=0, zmax=1, x=days_analyzed, y=top_patterns), row=row, col=col)
  fig.update_xaxes(title_text='days after occ', row=1, col=k+1)

title = f'Results of test for normality using different methods for top-{mode}-patterns'
fig.update_layout(template='plotly_dark', autosize=False, width=1000, height=400)
fig.update_yaxes(title_text='pattern', row=1, col=1)
fig.update_layout(title=title)
fig.update_xaxes(tickangle= -90)

fig.show()

In [21]:
# creating random dates for trading  (number of random trades equals number of occurences of a given pattern in a given mode)
# analyses is limited to the top_bull_patterns and top_bear_batterns

# depending on the mode, selction of top pattern and thier number of occurences
if mode == 'bullish':
  top_pattern = top_bull_patterns
  nb_occ = []
  for candle in top_patterns:
    nb_x = most_active_bull.loc[candle,'sum_bull']
    nb_occ.append(nb_x)
elif mode == 'bearish':
  top_pattern = top_bear_patterns
  nb_occ = []
  for candle in top_patterns:
    nb_x = most_active_bear.loc[candle,'sum_bear']
    nb_occ.append(nb_x)

# generation of a list of a random sequence of trade-times (expressed by index of prices),
# length of sequence equals number of occurences of a pattern in a mode
idx = []
for i, candle in enumerate(top_patterns):
  nb_xx = nb_occ[i]
  ind_rand = np.random.choice(range(len(prices)-days_look_ahead-1), nb_xx, replace=False)
  idx = idx + [ind_rand]

In [22]:
# trajectory dates for random trades (representing the trading frequency of certain patterns)
pt_id = 0
traject_bull_rand = pd.DataFrame()
traject_bear_rand = pd.DataFrame()

for i, candle in enumerate(tqdm(top_pattern)):
  idxx = idx[i].tolist()
  for id  in idxx:
    # preparations
    pt_id += 1
    # identification of the intervall between occurence and + day_look_ahead, calculation rets/cumrets
    id_x = prices.index[id: id + days_look_ahead + 1].date.tolist()
    pr_x = prices[prices.index.isin(id_x)]['Close']
    if mode == 'bullish':
      rets_x = pr_x.pct_change().fillna(0)          # since we are LONG
    elif mode == 'bearish':
      rets_x = - pr_x.pct_change().fillna(0)        # since we are SHORT
    cumrets_x = (1 + rets_x).cumprod()
    # transfering into a data-frame and connecting to the previous data
    dict_df =  {'pattern_rep': [candle] * (days_look_ahead + 1),
                'pattern_id':[pt_id] * (days_look_ahead + 1),
                'day': days,
                'close': pr_x.to_list(),
                'rets':rets_x.to_list(),
                'cumrets':cumrets_x.to_list()}
    tr_x_rand = pd.DataFrame(dict_df)
    if mode == 'bullish':
      traject_bull_rand = pd.concat([traject_bull_rand, tr_x_rand], axis=0)
    elif mode =='bearish':
      traject_bear_rand = pd.concat([traject_bear_rand, tr_x_rand], axis=0)

100%|██████████| 6/6 [00:04<00:00,  1.34it/s]


In [23]:
# comparison development of PnL (=cumrets) of random trades and the top-chart pattern
main_title = f'distribution of PnL of positions after occurences of {candle} in {mode} mode vs. random trade-entries'
title = main_title

# plotting
fig = make_subplots(rows=1, cols=5, shared_yaxes=True, subplot_titles = (selected_cols))

for i, day in enumerate(days_analyzed):
  if mode == 'bullish':
    tx_bull = traject_bull[(traject_bull['pattern'] == candle) & (traject_bull['day'] == day)]['cumrets']
    tx_bull_rand = traject_bull_rand[(traject_bull_rand['pattern_rep'] == candle) & (traject_bull_rand['day'] == day)]['cumrets']
    fig.add_trace(go.Histogram(y=tx_bull_rand.to_list(), name=f'random {days_str_analyzed[i]}', ybins=dict(start=0.75, end=1.25, size=0.01)), row=1, col=i+1)
    fig.add_trace(go.Histogram(y=tx_bull.to_list(), name=f'candle {days_str_analyzed[i]}', ybins=dict(start=0.75, end=1.25, size=0.01)), row=1, col=i+1)
    fig.add_hline(y=tx_bull.mean(), line_dash="dot", row=1, col=i+1, annotation_text=f'mean (candle): {tx_bull.mean():.4f}', annotation_position="bottom right")

  elif mode =='bearish':
    tx_bear = traject_bear[(traject_bear['pattern'] == candle) & (traject_bear['day'] == day)]['cumrets']
    tx_bear_rand = traject_bear_rand[(traject_bear_rand['pattern_rep'] == candle) & (traject_bear_rand['day'] == day)]['cumrets']
    fig.add_trace(go.Histogram(y=tx_bear_rand.to_list(), name=f'random {days_str_analyzed[i]}', ybins=dict(start=0.75, end=1.25, size=0.01)), row=1, col=i+1)
    fig.add_trace(go.Histogram(y=tx_bear.to_list(), name=f'candle {days_str_analyzed[i]}', ybins=dict(start=0.75, end=1.25, size=0.01)), row=1, col=i+1)
    fig.add_hline(y=tx_bear.mean(), line_dash="dot", row=1, col=i+1, annotation_text=f'mean (candle): {tx_bear.mean():.4f}', annotation_position="bottom right")

fig.update_layout(template='plotly_dark', autosize=False, width=1600, height=500)
fig.update_layout(title=title, xaxis_title='# of occurences', yaxis_title='PnL of the day', legend_title='days after occurence')
fig.update_layout(barmode='overlay')
fig.update_traces(opacity=0.65)
fig.show()

In [24]:
# Test for identity of two samples

def ident_distr_test(data1, data2, method, alpha):
  '''
  test of null-hypothesis: data1 and data 2 follows the same distribution
  Args:
    data1, data2: random samples to be anlyzed
    method: test method to be applied
    alpha: confidence level
  Returns:
    the test result: 1 in case the null-hypothesis is confirmed and 0 if null-hyothesis is rejected
  Raises:
    nothing
  '''
  res, statistic, pvalue = 0, 0, 0
  # perform Mann-Whitney U test
  if method == 'MannWhitney':
    statistic, pvalue = st.mannwhitneyu(data1, data2)
    res = 1 if pvalue > alpha else 0
  # perform Kolmogorov-Smirnov test
  if method == 'KolmogorovSmirnov':
    statistic, pvalue = st.ks_2samp(data1, data2)
    res = 1 if pvalue > alpha else 0
  # perform Cramer von Mieses test
  if method == 'CramervonMieses2samp':
    result = st.cramervonmises_2samp(data1, data2)
    res = 1 if result.pvalue > alpha else 0

  return (int(res))

In [25]:
# testing if PnL of positions, generated by candlesticks or generated by random entires follows the same distribution
# using different statistical tests, asessing top patterns and selected days

# applied tests
test_distr_names = ['KolmogorovSmirnov', 'MannWhitney', 'CramervonMieses2samp']

# preparations
if mode == 'bullish':
  top_patterns = top_bull_patterns
  traject_1 = traject_bull
  traject_2 = traject_bull_rand
elif mode == 'bearish':
  top_patterns = top_bear_patterns
  traject_1 = traject_bear
  traject_2 = traject_bear_rand
test_distr_results = []

# execution of tests for different candlsitckpatterns and days
for test in test_distr_names:
  distr_results = np.zeros((len(top_patterns), len(days_analyzed)))
  for i, candle in enumerate(top_patterns):
    tr1 = traject_1[traject_1['pattern'] == candle]
    tr2 = traject_2[traject_2['pattern_rep'] == candle]
    for j, day in enumerate(days_analyzed):
      data1 = tr1[tr1['day'] == day]['cumrets']
      data2 = tr2[tr2['day'] == day]['cumrets']
      distr_results[i,j] = ident_distr_test(data1, data2, test, alpha)
  test_distr_results =  test_distr_results + [distr_results]

In [26]:
# generation of an overview-plot about the test results for the top-patterns using different methods

# preparation
test_distr_str = ['Test: ' + str(test_distr_names[i]) for i in range(len(test_distr_names ))]
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, shared_xaxes=True, subplot_titles = (test_distr_str))

if mode == 'bullish':
  top_patterns = top_bull_patterns
elif mode == 'bearish':
  top_patterns = top_bear_patterns

#sub-plotting
for k, test in enumerate(test_distr_names):
  col = k + 1
  fig.add_trace(go.Heatmap(z=test_distr_results[k], zmin=0, zmax=1, x=days_analyzed, y=top_patterns), row=1, col=col)
  fig.update_xaxes(title_text='days after occ', row=1, col=col)

title = f'Test of identity of distribution of PnL after occ. of top-patterns, {mode} mode vs. random entries'
fig.update_layout(template='plotly_dark', autosize=False, width=1200, height=600)
fig.update_yaxes(title_text='pattern', row=1, col=1)
fig.update_layout(title=title)
fig.update_xaxes(tickangle= -90)

fig.show()